# Units and unit validation

`xsnow` uses SI units for representing data values: lengths such as snow height, layer height, grain size, and depth are stored in metres; stresses are stored in Pa; and dimensionless quantities are stored as `1`, not `%`. The main intentional exception is temperature, which is kept in degrees Celsius (`degC`) for its relevance with snow.

## Inspecting and converting units
The current unit of a variable is stored in `xs[var].attrs["units"]`. The underlying xsnow unit convention is stored in the [registry](../api/_generated/xsnow.registry), together with a `plot_units` entry that suggests meaningful units for plotting. `xsnow`'s [`UnitsAccessor`](../api/_generated/xsnow.accessors) allows users to easily access those units and convert between them.

In [1]:
import xsnow
xs = xsnow.single_profile_timeseries()

[i] xsnow.io: Parsing 2 datasets eagerly with 13 workers...
[i] xsnow.utils: Slope coordinate 'inclination' varies by location. Preserving (location, slope) dimensions as allow_per_location=True.
[i] xsnow.utils: Slope coordinate 'azimuth' varies by location. Preserving (location, slope) dimensions as allow_per_location=True.


In [2]:
# get current units of DataArray
xs["grain_size"].units.current

'm'

In [3]:
# get units recommended for plotting this DataArray
xs["grain_size"].units.plot

'mm'

Unit attributes are metadata. Manually editing `xs[var].attrs["units"]` does not convert the data values. To **convert units**, use the `UnitsAccessor` methods, see box below.

```{admonition} Converting between units
:class: note

| `UnitsAccessor` method | Example | Description |
| --- | --- | --- |
| `.units.to()` | `xs['grain_size'].units.to("cm")` | Convert to explicit units |
| `.units.to_plot_unit()` | `xs['grain_size'].units.to_plot_unit()` | Convert to recommended plotting units |
```

## Validating units
### upon parsing and writing
By default, `xsnow.read(..., unit_validation="strict")` validates unit metadata and converts values to xsnow units. For PRO files, strict mode checks all file headers. For SMET files, unit metadata is parsed per file. Incompatible units raise an error. For large homogeneous PRO directories, `unit_validation="light"` can be faster because it validates only one PRO file per parent directory. Use this only when you know the archive is internally consistent. NetCDF and Zarr stores are expected to already contain xsnow units. In strict mode, conflicting stored `attrs["units"]` raise; in light mode, they warn and are preserved unchanged to allow users to maintain their preferred units.

When `xsnowDataset`s are written back to file, their units and unit attributes remain untouched for NetCDF and Zarr dumps, but will be converted to existing SNOWPACK conventions for PRO and SMET files. SMET files can be written in input (i.e., forcing) convention or output convention. 

### during an analysis workflow
When you want to validate whether all values in your dataset are following the xsnow unit convention, you can

In [4]:
from xsnow.units import validate_xsnow_unit_attrs
validate_xsnow_unit_attrs(xs)

True